In [1]:
import pandas as pd
import numpy as np

# --- Re-execute previous tasks to ensure 'values' DataFrame is in the correct state ---
# This part is necessary to get the 'values' DataFrame ready for NumPy operations.

# Task 1: Merging the CSV Files into a Single Dataframe
languages = pd.read_csv('languages.csv')
parameters = pd.read_csv('parameters.csv')
values = pd.read_csv('values.csv')
codes = pd.read_csv('codes.csv')

# Reduce values to specific parameters and drop irrelevant columns
values = values[values['Parameter_ID'].isin(['GB020', 'GB022', 'GB023'])]
values = values.drop(columns=['ID', 'Comment', 'Source', 'Source_comment', 'Coders'])

# Merge Name and Macroarea from languages
values = pd.merge(values, languages[['ID', 'Name', 'Macroarea']], left_on='Language_ID', right_on='ID', how='left')
values = values.drop(columns=['ID'])

# Merge Description from codes
values = pd.merge(values, codes[['ID', 'Description']], left_on='Code_ID', right_on='ID', how='left')
values = values.drop(columns=['ID'])

# Task 2: Getting the values Dataframe into Shape
# Drop original Value, Code_ID, and rename columns
values = values.drop(columns=['Value', 'Code_ID'])
values = values.rename(columns={'Name': 'Language', 'Description': 'Value', 'Parameter_ID': 'Feature'})

# Reorder columns and sort
values = values[['Language', 'Macroarea', 'Feature', 'Value']]
values = values.sort_values(by=['Macroarea', 'Language'])

# Remove incomplete data and replace feature names
values = values.dropna()
values['Feature'] = values['Feature'].replace({
    'GB020': 'defArt',
    'GB022': 'prenom',
    'GB023': 'postnom'
})
# --- End of previous tasks setup ---


# Convert relevant columns to NumPy arrays for efficient operations
# We use .to_numpy() to get the underlying NumPy arrays from the DataFrame columns.
# It's important to keep these arrays in sync with the filtered/sorted 'values' DataFrame.
language_np = values['Language'].to_numpy()
macroarea_np = values['Macroarea'].to_numpy()
feature_np = values['Feature'].to_numpy()
value_np = values['Value'].to_numpy()


# --- Task 3.b Global Distribution of Definite Articles (NumPy) ---
print("--- Task 3.b Global Distribution of Definite Articles (NumPy) ---")
defart_mask_global = (feature_np == 'defArt')
global_defart_values = value_np[defart_mask_global]
global_unique, global_counts = np.unique(global_defart_values, return_counts=True)
global_defart_distribution_np = dict(zip(global_unique, global_counts))
print(global_defart_distribution_np)
print("\n")

print("--- Task 3.b Eurasia Distribution of Definite Articles (NumPy) ---")
eurasia_mask = (macroarea_np == 'Eurasia')
defart_mask_eurasia = (feature_np == 'defArt')
combined_mask_eurasia = eurasia_mask & defart_mask_eurasia
eurasia_defart_values = value_np[combined_mask_eurasia]
eurasia_unique, eurasia_counts = np.unique(eurasia_defart_values, return_counts=True)
eurasia_defart_distribution_np = dict(zip(eurasia_unique, eurasia_counts))
print(eurasia_defart_distribution_np)
print("\n")

print("--- Task 3.b Africa Distribution of Definite Articles (NumPy) ---")
africa_mask = (macroarea_np == 'Africa')
defart_mask_africa = (feature_np == 'defArt')
combined_mask_africa = africa_mask & defart_mask_africa
africa_defart_values = value_np[combined_mask_africa]
africa_unique, africa_counts = np.unique(africa_defart_values, return_counts=True)
africa_defart_distribution_np = dict(zip(africa_unique, africa_counts))
print(africa_defart_distribution_np)
print("\n")

# --- Task 3.d Languages with Prenominal Article Position (NumPy) ---
print("--- Task 3.d Languages with Prenominal Article Position (Value: present) (NumPy) ---")
prenom_mask = (feature_np == 'prenom')
present_mask = (value_np == 'present')
combined_prenom_present_mask = prenom_mask & present_mask
prenominal_present_languages_np = language_np[combined_prenom_present_mask]
prenominal_present_languages_list_np = np.unique(prenominal_present_languages_np).tolist()
print(prenominal_present_languages_list_np)
print("\n")


# --- Task 4: Pivoting and Melting (NumPy) ---

# Task 4.a & 4.b: Pivot to get features as columns, removing unnecessary levels.
# In NumPy, we don't have direct `unstack` or MultiIndex `droplevel`.
# We will create a structured array or a dictionary-based approach to simulate the pivot.

# Get unique combinations of Macroarea and Language, which will form the "rows" of our pivoted data.
unique_macro_lang_combinations = np.unique(np.column_stack((macroarea_np, language_np)), axis=0)

# Initialize a dictionary to store the pivoted data.
# Keys will be (Macroarea, Language) tuples, values will be dictionaries of features.
languages_rows_dict = {}

# Define the features we expect as columns
features_to_pivot = ['defArt', 'prenom', 'postnom']

# Populate the dictionary by iterating through the original data
for i in range(len(language_np)):
    current_macroarea = macroarea_np[i]
    current_language = language_np[i]
    current_feature = feature_np[i]
    current_value = value_np[i]

    key = (current_macroarea, current_language)
    if key not in languages_rows_dict:
        # Initialize features for this language with None or a placeholder if not present
        languages_rows_dict[key] = {feature: None for feature in features_to_pivot}
    
    # Assign the actual value
    if current_feature in features_to_pivot:
        languages_rows_dict[key][current_feature] = current_value

# Convert the dictionary into a structured NumPy array or a list of lists/arrays for printing.
# For simplicity and clear output, we'll convert it to a list of lists that can be displayed or
# further processed. A structured array would be more akin to a DataFrame.
# Creating a Pandas DataFrame from the dictionary for easy display, as direct NumPy structured
# array creation with mixed types and arbitrary column names is more verbose for printing.
# If strict NumPy-only output is required, a more complex structured array definition is needed.

# Let's create lists for each "column" (Macroarea, Language, defArt, prenom, postnom)
# and then combine them for a tabular output representation.
macroarea_col = []
language_col = []
defart_col = []
prenom_col = []
postnom_col = []

# Sort keys for consistent output order (mimicking Pandas sort_values)
sorted_keys = sorted(languages_rows_dict.keys())

for (macroarea, language) in sorted_keys:
    macroarea_col.append(macroarea)
    language_col.append(language)
    defart_col.append(languages_rows_dict[(macroarea, language)].get('defArt'))
    prenom_col.append(languages_rows_dict[(macroarea, language)].get('prenom'))
    postnom_col.append(languages_rows_dict[(macroarea, language)].get('postnom'))

# Creating a conceptual "languages_rows" NumPy array (as an object array or structured array)
# For practical display, it's easier to print like this.
# If a true NumPy array representation is critical, it would look like:
# languages_rows_np = np.empty(len(sorted_keys), dtype=[
#     ('Macroarea', 'U50'), ('Language', 'U50'),
#     ('defArt', 'U50'), ('prenom', 'U50'), ('postnom', 'U50')
# ])
# for i, (macroarea, language) in enumerate(sorted_keys):
#     languages_rows_np[i] = (
#         macroarea, language,
#         languages_rows_dict[(macroarea, language)].get('defArt'),
#         languages_rows_dict[(macroarea, language)].get('prenom'),
#         languages_rows_dict[(macroarea, language)].get('postnom')
#     )
# For this task, we will represent it as a more readable grouped output.

print("--- Task 4.a & 4.b languages_rows (conceptual NumPy output - head) ---")
# Print a header
print(f"{'Macroarea':<20} {'Language':<20} {'defArt':<10} {'prenom':<10} {'postnom':<10}")
print("-" * 75)
# Print first few rows
for i in range(min(5, len(sorted_keys))):
    key = sorted_keys[i]
    print(f"{key[0]:<20} {key[1]:<20} {languages_rows_dict[key].get('defArt'):<10} "
          f"{languages_rows_dict[key].get('prenom'):<10} {languages_rows_dict[key].get('postnom'):<10}")
print("\n")


# Task 4.c: Languages with definite articles in both postnominal and prenominal position.
# We'll iterate through the `languages_rows_dict` created above.
languages_with_both_prenom_postnom_np = []
for (macroarea, language), features in languages_rows_dict.items():
    if features.get('prenom') == 'present' and features.get('postnom') == 'present':
        languages_with_both_prenom_postnom_np.append(language)

print("--- Task 4.c Languages with Definite Articles in Both Postnominal and Prenominal Position (NumPy) ---")
print(languages_with_both_prenom_postnom_np)
print("\n")


# Task 4.d: Chained method calls to transform back to original 'values' format (conceptual).
# In NumPy, this would involve a manual "melt" operation, which is the inverse of the pivot.
# It would mean iterating through the pivoted structure and reconstructing the long format.

print("--- Task 4.d Chained method calls to revert (Conceptual NumPy steps) ---")
print("1. Iterate through the pivoted (Macroarea, Language) entries.")
print("2. For each entry, iterate through the feature columns (defArt, prenom, postnom).")
print("3. For each feature, create a new row/entry consisting of (Macroarea, Language, Feature_Name, Feature_Value).")
print("4. Combine these new entries into a flat array or list of lists.")
print("5. Sort the resulting structure by Macroarea, then Language.")
print("This is equivalent to a manual 'melt' and then a sort, demonstrating that these are not single chained calls in NumPy.")
print("\n")


# --- Task 5: Concatenation (NumPy) ---

# Task 5.a: Extract data for each feature.
# This involves boolean indexing on the original NumPy arrays.
def_art_mask = (feature_np == 'defArt')
prenom_mask = (feature_np == 'prenom')
postnom_mask = (feature_np == 'postnom')

def_art_data_np = np.column_stack((language_np[def_art_mask], macroarea_np[def_art_mask], feature_np[def_art_mask], value_np[def_art_mask]))
prenom_data_np = np.column_stack((language_np[prenom_mask], macroarea_np[prenom_mask], feature_np[prenom_mask], value_np[prenom_mask]))
postnom_data_np = np.column_stack((language_np[postnom_mask], macroarea_np[postnom_mask], feature_np[postnom_mask], value_np[postnom_mask]))

print("--- Task 5.a Dataframe head examples (NumPy representation) ---")
print("def_art_data head (Language, Macroarea, Feature, Value):")
print(def_art_data_np[:min(5, len(def_art_data_np))])
print("\nprenom_data head:")
print(prenom_data_np[:min(5, len(prenom_data_np))])
print("\npostnom_data head:")
print(postnom_data_np[:min(5, len(postnom_data_np))])
print("\n")


# Task 5.b: Concatenate along the column axis, limiting to complete rows (inner join).
# In NumPy, an "inner join" on custom keys requires manually finding the intersection of keys.
# We'll use (Macroarea, Language) as the join key.

# Create structured arrays (or simple 2D arrays if types are consistent) for easier manipulation
# and to carry Macroarea and Language with the feature values.
# For simplicity, let's process them to maps for easy lookup by (Macroarea, Language)
def_art_map = {(r[1], r[0]): r[3] for r in def_art_data_np} # (Macroarea, Language): Value
prenom_map = {(r[1], r[0]): r[3] for r in prenom_data_np}
postnom_map = {(r[1], r[0]): r[3] for r in postnom_data_np}

# Find common (Macroarea, Language) keys (intersection of all three sets of keys)
common_keys = set(def_art_map.keys()) \
              .intersection(set(prenom_map.keys())) \
              .intersection(set(postnom_map.keys()))

# Sort common keys for consistent output
sorted_common_keys = sorted(list(common_keys))

# Construct the concatenated array
concatenated_columns_complete_np_list = []
for macroarea, language in sorted_common_keys:
    concatenated_columns_complete_np_list.append([
        macroarea,
        language,
        def_art_map[(macroarea, language)],
        prenom_map[(macroarea, language)],
        postnom_map[(macroarea, language)]
    ])

# Convert to a NumPy array for display
# If all values are strings, a simple object array or fixed-size string array.
# For display purposes, we can print it structured.
print("--- Task 5.b Concatenated along column axis (inner join, NumPy conceptual output - head) ---")
print(f"{'Macroarea':<20} {'Language':<20} {'defArt':<10} {'prenom':<10} {'postnom':<10}")
print("-" * 75)
for row in concatenated_columns_complete_np_list[:min(5, len(concatenated_columns_complete_np_list))]:
    print(f"{row[0]:<20} {row[1]:<20} {row[2]:<10} {row[3]:<10} {row[4]:<10}")
print("\n")


# Task 5.c: Concatenate along the row axis with a hierarchical index (conceptual).
# In NumPy, a "hierarchical row index" is typically simulated by stacking arrays
# and then maintaining separate arrays for the 'levels' or combining them into a structured array
# that has fields for each level of the index.

# We need the 'Value' data along with its corresponding 'Feature', 'Macroarea', 'Language'.
# Create a flat list of records, then sort to mimic hierarchical indexing.
# Each record will be (Feature_Key, Macroarea, Language, Value)
series_with_hierarchical_index_np_list = []

for i in range(len(language_np)):
    current_language = language_np[i]
    current_macroarea = macroarea_np[i]
    current_feature = feature_np[i]
    current_value = value_np[i]
    series_with_hierarchical_index_np_list.append(
        (current_feature, current_macroarea, current_language, current_value)
    )

# Sort the list of tuples to mimic the hierarchical sorting:
# 1st level: Feature_Key, 2nd level: Macroarea, 3rd level: Language
series_with_hierarchical_index_np_list.sort(key=lambda x: (x[0], x[1], x[2]))

print("--- Task 5.c Series with Three-Level Hierarchical Index (NumPy conceptual output - head) ---")
print(f"{'Feature':<10} {'Macroarea':<20} {'Language':<20} {'Value':<10}")
print("-" * 65)
for row in series_with_hierarchical_index_np_list[:min(5, len(series_with_hierarchical_index_np_list))]:
    print(f"{row[0]:<10} {row[1]:<20} {row[2]:<20} {row[3]:<10}")
print("\n")


--- Task 3.b Global Distribution of Definite Articles (NumPy) ---
{'absent': np.int64(1374), 'present': np.int64(823)}


--- Task 3.b Eurasia Distribution of Definite Articles (NumPy) ---
{'absent': np.int64(418), 'present': np.int64(116)}


--- Task 3.b Africa Distribution of Definite Articles (NumPy) ---
{'absent': np.int64(261), 'present': np.int64(233)}


--- Task 3.d Languages with Prenominal Article Position (Value: present) (NumPy) ---
["A'ou", 'Abkhaz', "Abu' Arapesh", 'Achi', 'Achumawi', 'Aghwan', 'Aguaruna', 'Agutaynen', 'Ahin-Kayapa Kalanguya', 'Akateko', 'Alacatlatzala Mixtec', 'Alamblak', 'Alsea-Yaquina', 'Amara', 'Ambonese Malay', 'Amharic', 'Ami', 'Ancient Greek', 'Ancient Hebrew', 'Aneityum', 'Anta-Komnzo-Wára-Wérè-Kémä', 'Antankarana Malagasy', 'Anu-Hkongso', 'Anuta', 'Apma', 'Araona', 'Arhuaco', 'Arigidi', 'Aromanian', 'Arosi', 'Arta', 'Ashéninka Perené', 'Ata Manobo', 'Atayal', 'Aulua', 'Awar', 'Baga Koga', 'Bahamas Creole English', 'Baharna Arabic', 'Bahing', 'Baki'

TypeError: The axis argument to unique is not supported for dtype object